In [ ]:
!uv pip install pybest

In [ ]:
from pybest.gbasis import (
    compute_cholesky_eri,
    compute_kinetic,
    compute_nuclear,
    compute_nuclear_repulsion,
    compute_overlap,
    get_gobasis,
)
from pybest.linalg import CholeskyLinalgFactory
from pybest.localization import PipekMezey
from pybest.occ_model import AufbauOccModel
from pybest.part import get_mulliken_operators
from pybest.wrappers import RHF

xyz = """6
        Ethylene
        C    0.000    0.000    0.667
        C    0.000    0.000   -0.667
        H    0.000    0.922    1.237
        H    0.000   -0.922    1.237
        H    0.000    0.922   -1.237
        H    0.000   -0.922   -1.237
#     """

with open("mol.xyz", "w") as file:
    file.write(xyz)

# get the XYZ file from PyBEST's test data directory
basis = get_gobasis("cc-pcvdz", "mol.xyz")

# Define Occupation model, orbitals, and overlap
lf = CholeskyLinalgFactory(basis.nbasis)
occ_model = AufbauOccModel(basis, ncore=0)
orb_a = lf.create_orbital(basis.nbasis)
olp = compute_overlap(basis)

# Construct Hamiltonian
kin = compute_kinetic(basis)
ne = compute_nuclear(basis)
eri = compute_cholesky_eri(basis, threshold=1e-8)
nr = compute_nuclear_repulsion(basis)

# Do a Hartree-Fock calculation
hf = RHF(lf, occ_model)
hf_output = hf(kin, ne, eri, nr, olp, orb_a)

# Localize orbitals to improve pCCD convergence
mulliken = get_mulliken_operators(basis)
loc = PipekMezey(lf, occ_model, mulliken)
loc(orb_a, "occ")
loc(orb_a, "virt")

In [75]:
basis_gto = BasisGTO.from_pybest_basis(basis)

In [ ]:
plot_molecular_orbital(
        basis_gto,
        # orb_a.coeffs,
        spherical_to_cartesian_matrix(orb_a.coeffs, basis.shell_types),
        n=3,
        isovalue=0.07,
        title=f"Molecule",
    )

ValueError: mo_coeffs has 48 rows but basis has 50 functions.

In [ ]:
import numpy as np

def build_cart2sph_matrix(shell_angular_momenta):
    """
    Builds the block-diagonal transformation matrix T to convert 
    Cartesian AOs to Spherical AOs.
    
    Args:
        shell_angular_momenta (list of int): List of l-quantum numbers 
                                             for each shell (e.g., [0, 1, 0, 2] for s, p, s, d)
    Returns:
        np.ndarray: Transformation matrix T of shape (N_sph, N_cart)
    """
    blocks = []
    
    # Pre-defined transformation blocks for s, p, d (normalized Cartesian to normalized Spherical)
    # Ordering assumptions below must match your specific quantum chemistry package!
    
    # s-functions (1x1)
    T_s = np.array([[1.0]])
    
    # p-functions (3x3): Assuming (x, y, z) -> (x, y, z)
    T_p = np.eye(3)
    
    # d-functions (5x6): 
    # Cartesian order: xx, yy, zz, xy, xz, yz
    # Spherical order: xy, yz, z^2, xz, x^2-y^2
    # Note: Coefficients account for the normalization of Cartesian GTOs.
    sqrt3 = np.sqrt(3.0)
    T_d = np.array([
        [0.0,  0.0,  0.0,  1.0,  0.0,  0.0],           # xy
        [0.0,  0.0,  0.0,  0.0,  0.0,  1.0],           # yz
        [-0.5, -0.5, 1.0,  0.0,  0.0,  0.0],           # z^2
        [0.0,  0.0,  0.0,  0.0,  1.0,  0.0],           # xz
        [sqrt3/2, -sqrt3/2, 0.0, 0.0,  0.0,  0.0]      # x^2 - y^2
    ])

    for l in shell_angular_momenta:
        if l == 0:
            blocks.append(T_s)
        elif l == 1:
            blocks.append(T_p)
        elif l == 2:
            blocks.append(T_d)
        else:
            raise NotImplementedError(f"Transformation for l={l} (f-orbitals and beyond) not implemented here.")
            
    # Assemble the full block-diagonal matrix using scipy or numpy
    from scipy.linalg import block_diag
    T_full = block_diag(*blocks)
    
    return T_full

def convert_mo_matrix_cart2sph(C_cart, shell_angular_momenta):
    """
    Converts a Cartesian MO coefficient matrix to a Spherical one.
    
    Args:
        C_cart (np.ndarray): MO coefficients in Cartesian basis (N_cart_AOs, N_MOs)
        shell_angular_momenta (list of int): List of l-quantum numbers for the basis.
        
    Returns:
        np.ndarray: MO coefficients in Spherical basis (N_sph_AOs, N_MOs)
    """
    T = build_cart2sph_matrix(shell_angular_momenta)
    
    # Apply transformation: C_sph = T @ C_cart
    C_sph = np.dot(T, C_cart)
    
    return C_sph

In [18]:
from pybest.cc import RfpCCSD
from pybest.geminals import ROOpCCD
from pybest.gbasis import (
    compute_cholesky_eri,
    compute_kinetic,
    compute_nuclear,
    compute_nuclear_repulsion,
    compute_overlap,
    get_gobasis,
)
from pybest.linalg import CholeskyLinalgFactory
from pybest.localization import PipekMezey
from pybest.occ_model import AufbauOccModel
from pybest.part import get_mulliken_operators
from pybest.wrappers import RHF


def compute_for_r(r: float, orb_a=None):

    print(f"\n >>>Current radius: {r}")
    xyz = f"""2						
                
    C	0.0	0.0	 -{r/2}
    C	0.0	0.0	 {r/2}
    """
    with open("mol.xyz", "w") as file:
        file.write(xyz)

    # get the XYZ file from PyBEST's test data directory
    basis = get_gobasis("cc-pcvdz", "mol.xyz")
    lf = CholeskyLinalgFactory(basis.nbasis)
    occ_model = AufbauOccModel(basis, ncore=0)
    olp = compute_overlap(basis)

    # Hamiltoniam
    kin = compute_kinetic(basis)
    ne = compute_nuclear(basis)
    eri = compute_cholesky_eri(basis, threshold=1e-8)
    nr = compute_nuclear_repulsion(basis)

    # HF
    if not orb_a:
        orb_a = lf.create_orbital(basis.nbasis)
        hf = RHF(lf, occ_model)
        hf_output = hf(kin, ne, eri, nr, olp, orb_a)

        # Localize orbitals to improve pCCD convergence
        mulliken = get_mulliken_operators(basis)
        loc = PipekMezey(lf, occ_model, mulliken)
        loc(orb_a, "occ")
        loc(orb_a, "virt")

        # Do pCCD
        pccd = ROOpCCD(lf, occ_model)
        pccd_output = pccd(
            hf_output,
            kin, ne, eri, e_core=nr,
            maxiter={"orbiter": 1000},
            )
    else:
        # Do pCCD
        pccd = ROOpCCD(lf, occ_model)
        pccd_output = pccd(
            kin, ne, eri, olp, e_core=nr,
            maxiter={"orbiter": 1000},
            orb_a=orb_a,
            )
        hf_output=None

    # Do pCCD-fpCCSD calculation
    if pccd_output.converged:
        ccsd = RfpCCSD(lf, occ_model)
        ccsd_output = ccsd(pccd_output, kin, ne, eri, solver="krylov")
    else:
        raise ValueError("pCCD not converged")
    return {
        "r": r,
        "orb_a": orb_a,
        "hf": hf_output,
        "pccd": pccd_output,
        "ccsd": ccsd_output,
    }

In [ ]:
results = compute_for_r(2.4)

In [13]:
results["orb_a"]

In [21]:
radius = [2.35, 2.3, 2.25, 2.2, 2.15, 2.1, 2.05, 2, 1.95, 1.9, 1.85, 1.8]
all_results = [results]

for r in radius:
    out = compute_for_r(r, orb_a=all_results[-1]["orb_a"])
    all_results.append(out)


 >>>Current radius: 2.35
####################################################################################################
Printing basis for whole molecule:
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Occupation Module:
 
Aufbau occupation number model.
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
              Total number of electrons: 12
                Total electronic charge: 0
                                         alpha
        Total number of basis functions: 36   
      Total number of occupied orbitals: 6    
       Total number of virtual orbitals: 30   
         Number of frozen core orbitals: 0    
   Number of active core orbitals (CVS): 0    
     Number of active occupied orbitals: 6    
      Number of active virtual orbitals: 30   
              Numbe

ValueError: pCCD not converged

In [22]:
len(all_results)

7

In [23]:
radius = [2.45, 2.5, 2.55, 2.6, 2.65, 2.7, 2.8, 2.9, 3.0, 3.2, 3.4, 3.6, 3.8, 4.0, 4.2, 4.4, 4.6, 4.8, 5.0, 5.5, 6.0]
all_results = [results]

for r in radius:
    out = compute_for_r(r, orb_a=all_results[0]["orb_a"])
    all_results = [out] + all_results


 >>>Current radius: 2.45
####################################################################################################
Printing basis for whole molecule:
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Occupation Module:
 
Aufbau occupation number model.
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
              Total number of electrons: 12
                Total electronic charge: 0
                                         alpha
        Total number of basis functions: 36   
      Total number of occupied orbitals: 6    
       Total number of virtual orbitals: 30   
         Number of frozen core orbitals: 0    
   Number of active core orbitals (CVS): 0    
     Number of active occupied orbitals: 6    
      Number of active virtual orbitals: 30   
              Numbe

ValueError: pCCD not converged